In [1]:
%reload_ext autoreload
%autoreload 2

# Verification that multiple corpus for same query is handles by sbert

In [3]:
from sentence_transformers.evaluation import InformationRetrievalEvaluator
import json

# build evaluator (same as before)
queries = {"q1": "example query 1"}
corpus = {"c1": "doc1", "c2": "doc2", "c4": "doc4", "c6": "doc6", "c8": "doc8"}
relevant_docs = {"q1": {"c1", "c2"}}

ks = [1, 3, 5]
evaluator = InformationRetrievalEvaluator(queries=queries, corpus=corpus, relevant_docs=relevant_docs, show_progress_bar=False, name="verify_mrr", 
                                        mrr_at_k=ks,
                                        ndcg_at_k=ks,
                                        accuracy_at_k=ks,
                                        precision_recall_at_k=ks,
                                        map_at_k=ks,
            )

# your predictions dict (query_id -> list of {corpus_id, score})
preds_dict = {
    "q1": [
        {"corpus_id": "c4", "score": 0.9},
        {"corpus_id": "c6", "score": 0.8},
        {"corpus_id": "c1", "score": 0.7},
        {"corpus_id": "c8", "score": 0.6},
        {"corpus_id": "c2", "score": 0.5},
    ]
}

# Convert to ordered list expected by compute_metrics
preds_ordered = [preds_dict.get(qid, []) for qid in evaluator.queries_ids]

# Compute metrics
metrics = evaluator.compute_metrics(preds_ordered)
print(metrics)
# manual check: first relevant at rank 3 -> MRR = 1/3 = 0.333333...

{'accuracy@k': {1: 0.0, 3: 1.0, 5: 1.0}, 'precision@k': {1: np.float64(0.0), 3: np.float64(0.3333333333333333), 5: np.float64(0.4)}, 'recall@k': {1: np.float64(0.0), 3: np.float64(0.5), 5: np.float64(1.0)}, 'ndcg@k': {1: np.float64(0.0), 3: np.float64(0.3065735963827292), 5: np.float64(0.5437713091520254)}, 'mrr@k': {1: 0.0, 3: 0.3333333333333333, 5: 0.3333333333333333}, 'map@k': {1: np.float64(0.0), 3: np.float64(0.16666666666666666), 5: np.float64(0.3666666666666667)}}


In [8]:
# getting the query file
# load csv from github
import pandas as pd

queries_df = pd.read_csv("https://raw.githubusercontent.com/NASA-IMPACT/akd-evals/refs/heads/master/documents/code_validation_data_v2.csv?token=GHSAT0AAAAAADJD3G5WDXNS6N5CLXD4RZDK2JPIRRA")

queries_df

,question,url,division
0,How do I create PDS labels?,"['https://github.com/NASA-AMMOS/labelocity', '...",Planetary
1,"How do I create a simulated mission, from star...",['https://github.com/NASA-AMMOS/aerie-mission-...,Planetary
2,Is my work compliant and ready to be uploaded ...,['https://github.com/NASA-PDS/data-upload-mana...,Planetary
3,Provide Jupyter notebook examples illustrating...,['https://github.com/Caltech-IPAC/MontageMosai...,Astro
4,Provide Python examples for analysis of Fermi ...,['https://github.com/fermi-lat/AnalysisThreads...,Astro
...,...,...,...
210,What pipeline ingests Sentinel-1 scenes and pe...,['https://github.com/aria-jpl/s1_qc_ingest'],Earth
211,Where can I find a script that tracks new Sent...,['https://github.com/aria-jpl/scihub_acquisiti...,Earth
212,Is there a converter to turn Sentinel-1 or oth...,['https://github.com/aria-jpl/slcp2cod'],Earth
213,How can I transform sparse coherence (slcp) pr...,['https://github.com/aria-jpl/slcp2cor'],Earth


In [9]:
corpus_df = pd.read_csv("/rhome/sawale/indus_traning/sentense_transformers/data/code_dataset/repositories_with_embeddings_v6.csv")
corpus_df.head()

,URL,text,reformulated_text,key_topics,area,reasoning,source,readme_url,description,name,relevant_content,relevant_reasoning,embeddings
0,https://github.com/Caltech-IPAC/Kepler-Discove...,Kepler-Discoveries ====== Software repository ...,Kepler-Discoveries is a software repository de...,Kepler Mission | NExScI | NASA Exoplanet Archi...,Astrophysics Division,The README content mentions the Kepler Mission...,Org,https://github.com/Caltech-IPAC/Kepler-Discove...,Utility scripts using the NASA Exoplanet Archi...,Kepler-Discoveries,,,"-0.0074941935,0.024302335,0.0007752162,-0.0270..."
1,https://github.com/Caltech-IPAC/Montage,"Montage: Astronomical Image Mosaics, Examinati...",Montage is an open-source toolkit designed for...,Montage | astronomical image mosaics | FITS im...,Astrophysics Division,"The README describes Montage, a toolkit for as...",Org,https://github.com/Caltech-IPAC/Montage/blob/m...,Image Mosaics for Astronomers,Montage,,,"0.005070961,-0.0024839384,-0.028777495,-0.0232..."
2,https://github.com/Caltech-IPAC/MontageMosaics,Jupyter Notebooks for Building Astronomical Mo...,Montage is an open-source toolkit licensed und...,Montage | astronomical mosaics | image process...,Astrophysics Division,"The README describes Montage, a toolkit for cr...",Org,https://github.com/Caltech-IPAC/MontageMosaics...,,MontageMosaics,,,"0.005882572,0.01452603,-0.009796207,-0.026566,..."
3,https://github.com/Caltech-IPAC/MontageNotebooks,Montage Notebooks: Jupyter notebooks illustrat...,Montage Notebooks provide Jupyter notebooks th...,Montage Notebooks | Jupyter notebooks | astron...,Astrophysics Division,"The README describes Montage, a toolkit for as...",Org,https://github.com/Caltech-IPAC/MontageNoteboo...,Jupyter notebooks illustrating the use of the ...,MontageNotebooks,,,"0.02123165,-0.0016249721,0.0004058171,-0.02378..."
4,https://github.com/Caltech-IPAC/TESSCoadds,"# <font color=""#880000""> Image Cutouts of Co-A...",Image Cutouts of Co-Added TESS Full Frame Imag...,TESS | Transiting Exoplanet Sky Survey | CCD c...,Astrophysics Division,The README describes the Transiting Exoplanet ...,Org,https://github.com/Caltech-IPAC/TESSCoadds/blo...,,TESSCoadds,,,"-0.012586548,0.00053917244,-0.005944926,-0.013..."


In [10]:
# making a new cols _text with template

template = """
`Description`: {},

`Reformulated Text`: {},

`Key Topics`: {},

`Relevant Content`: {}
"""

# ensure no NaNs and convert to str
corpus_df = corpus_df.fillna("")
corpus_df['_text'] = corpus_df.apply(
    lambda row: template.format(
        row['description'],
        row['reformulated_text'],
        row['key_topics'],
        row['relevant_content']
    ),
    axis=1
)

In [13]:
corpus_df["_text"].iloc[0]

'\n`Description`: Utility scripts using the NASA Exoplanet Archive API in support of the Kepler Mission Discoveries website,\n\n`Reformulated Text`: Kepler-Discoveries is a software repository designed to support the NExScI (NASA Exoplanet Science Institute) in managing the Kepler Mission Discoveries website. This project includes two main components: 1. The names_table feature, which generates the primary table displayed on the Discoveries page by retrieving data from the NASA Exoplanet Archive, a comprehensive database of exoplanet information. 2. The flc feature, which creates XML (Extensible Markup Language) files that facilitate the display of folded light curves, also utilizing data sourced from the NASA Exoplanet Archive. This software aids in the visualization and analysis of exoplanet discoveries made by the Kepler mission, enhancing public access to this scientific data.,\n\n`Key Topics`: Kepler Mission | NExScI | NASA Exoplanet Archive | exoplanets | light curves | data visu

In [16]:
corpus_df["area"].value_counts()

area
Astrophysics Division                        2401
Earth Science Division                       2125
Planetary Science Division                    544
Biological and Physical Sciences Division     245
Heliophysics Division                         130
Name: count, dtype: int64